# Create Training Labels from the Critical Mineral Deposits Database
from shapefile to GeoTIFF


In [1]:
import os
import numpy as np
import geopandas as gpd

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.io import load_raster
from beak.utilities.conversions import create_binary_raster


# Load data

**User definitions**

In [6]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 2240
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "us_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")

# Points file and query to select relevant mineral occurences
PATH_SHAPEFILE = BASE_PATH / "RAW" / "mineral_deposits" / "Porphyry_Copper" / "Dicken_US" / "pCu_deps_pros_usgs_80_percent_us_cont.shp"
SQL_QUERY = None

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("national" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str(BASE_RASTER.stem + "_POCO_HM9_PCUDEPPRO_80P" + ".tif")
OUT_FILE = PATH_EXPORT

print(f"Output file: {OUT_FILE}")


Output file: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\PROCESSED\national_102008_2240_us_cont\labels\EPSG_102008_RES_2240_us_cont_POCO_PCUDEPPRO_HM9_80P.tif


# Create Labels


Reproject to **BASE RASTER** CRS

In [7]:
from beak.utilities.vector_processing import _reproject_vector_data

base_raster = load_raster(BASE_RASTER)
template_crs = base_raster.crs
gdf = _reproject_vector_data(PATH_SHAPEFILE, crs=template_crs, encoding="utf-8", query=SQL_QUERY)


Create binary label raster

In [8]:
out_array = create_binary_raster(gdf, base_raster, query=SQL_QUERY, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(out_array==1)}")

Number of positive training labels: 113
